# 教程: 结果分析 - 模型性能评估指标

## 目的
在将模拟结果与观测数据进行比较时，除了直观地观察流量过程线之外，我们还需要一套客观的、定量的**性能评估指标（Performance Metrics）**来评价模型的好坏。这些指标可以帮助我们判断模型的改进方向，或者在不同模型之间进行选择。

本教程将介绍几种在水文学中最常用的模型性能评估指标。

此笔记本将：
1.  加载一个包含模拟流量和观测流量的时间序列文件。
2.  实现并解释以下四种核心评估指标：
    - **纳什效率系数 (Nash-Sutcliffe Efficiency, NSE)**
    - **克林-古普塔效率系数 (Kling-Gupta Efficiency, KGE)**
    - **百分比偏差 (Percent Bias, PBIAS)**
    - **均方根误差 (Root Mean Square Error, RMSE)**
3.  计算这些指标的值，并解释它们的含义。
4.  绘制一个模拟值 vs. 观测值的散点图，作为一种直观的评估工具。

## 1. 环境设置和加载数据

我们使用`run_example.py`生成的`final_comparison_table.csv`文件，其中包含了配对的观测流量和模拟流量。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 加载数据
df = pd.read_csv('../examples/results/final_comparison_table.csv', index_col='date', parse_dates=True)
df = df.dropna() # 确保没有缺失值

sim = df['simulated_flow'].values
obs = df['observed_flow'].values

print("成功加载模拟值和观测值。")

## 2. 计算性能指标

我们为每种指标编写一个独立的计算函数。

In [ ]:
def calculate_nse(sim, obs):
    """纳什效率系数 (Nash-Sutcliffe Efficiency, NSE)
    范围: -∞ 到 1。1表示完美拟合。0表示模型与观测均值一样好。负值表示模型比观测均值还差。
    """
    numerator = np.sum((sim - obs) ** 2)
    denominator = np.sum((obs - np.mean(obs)) ** 2)
    return 1 - (numerator / denominator)

def calculate_kge(sim, obs):
    """克林-古普塔效率系数 (Kling-Gupta Efficiency, KGE)
    范围: -∞ 到 1。1表示完美拟合。它将误差分解为相关性、偏差和变率三部分。
    """
    r = np.corrcoef(sim, obs)[0, 1]
    beta = np.mean(sim) / np.mean(obs)
    gamma = (np.std(sim) / np.mean(sim)) / (np.std(obs) / np.mean(obs))
    return 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

def calculate_pbias(sim, obs):
    """百分比偏差 (Percent Bias, PBIAS)
    衡量模拟值总体上是偏高还是偏低。0是最优值。正值表示模型偏高，负值表示模型偏低。
    """
    return 100 * (np.sum(sim - obs) / np.sum(obs))

def calculate_rmse(sim, obs):
    """均方根误差 (Root Mean Square Error, RMSE)
    衡量误差的量级，单位与原始数据相同。0是最优值。
    """
    return np.sqrt(np.mean((sim - obs) ** 2))

# --- 计算所有指标 ---
nse = calculate_nse(sim, obs)
kge = calculate_kge(sim, obs)
pbias = calculate_pbias(sim, obs)
rmse = calculate_rmse(sim, obs)

print("--- 模型性能评估结果 ---")
print(f"NSE:   {nse:.3f}")
print(f"KGE:   {kge:.3f}")
print(f"PBIAS: {pbias:.2f}%")
print(f"RMSE:  {rmse:.2f} m³/s")

### 结果解读

- **NSE / KGE**: 这两个指标的值都非常低（远小于0，甚至是负数），表明模型的性能很差。这主要是因为这个`run_example`的参数是随意设置的，并未经过率定。
- **PBIAS**: 值为-84.51%，这是一个非常大的负偏差，说明模型**严重低估**了流量。
- **RMSE**: 均方根误差为48.69 m³/s，考虑到观测流量的范围（约10-150 m³/s），这是一个相当大的误差。

**结论**: 这些指标定量地告诉我们，这个未经率定的模型的性能是不可接受的，它严重低估了径流。这凸显了模型率定（如使用EnKF）的重要性。

## 3. 可视化评估: 散点图

除了数值指标，模拟值-观测值散点图也是一种非常直观的评估工具。我们将观测值作为X轴，模拟值作为Y轴，绘制散点图。一个完美的模型，所有的点都应该落在1:1对角线上。


In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(obs, sim, alpha=0.6, label='Simulated vs. Observed')

# 绘制1:1对角线作为参考
max_val = max(np.max(obs), np.max(sim))
plt.plot([0, max_val], [0, max_val], 'r--', label='1:1 Line')

plt.title('Simulated vs. Observed Flow')
plt.xlabel('Observed Discharge (m³/s)')
plt.ylabel('Simulated Discharge (m³/s)')
plt.legend()
plt.grid(True)
plt.axis('equal') # 保证X和Y轴的比例相同
plt.show()

print("从散点图中可以非常直观地看到，所有的点都位于1:1对角线的下方，再次印证了模型存在系统性的严重低估问题。")